In [12]:
import numpy as np
import pandas as pd
import random
from random import randint
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import xgboost as xgb
from util_game import *
from util_strategy import *

In [2]:
def get_params_DiamondChange():
    X = {}
    Y = {}
    Z = {}
    columns = ['y']
    columns.append('est')
    # change range, and column counts
    for j in range(-1,2):
        X[j]=[]
        Y[j]=[]
        Z[j]=pd.DataFrame(columns=columns)
    
    
    for i in range(100):
        game_value = get_game_value()
        last_yield = -1
        #
        sum_log_yield = 5.0*np.log(101.0)
        for j in range(150):
            picked, mining_time_first, mining_time = sample_picked_once_with_time(game_value)
            if j==0:
                change_of_gold = -1
                change_of_diamond = -1
                fifty_yields = []
                for k in range(50):
                    fifty_yields.append(sample_yield_once(game_value))
                last_lowest = 1e9
                last_highest = -1e9
                exact_diamond = -1
            else:
                fifty_yields = []
                change_of_gold = picked[2] - last_gold
                if change_of_gold<-3:
                    change_of_gold = -2
                elif change_of_gold<=-1:
                    change_of_gold = -1
                elif change_of_gold<=0:
                    change_of_gold = 0
                elif change_of_gold<=3:
                    change_of_gold = 1
                else:
                    change_of_gold = 2
                change_of_diamond = picked[4] - last_diamond
                if change_of_diamond<-1:
                    change_of_diamond = -1
                elif change_of_diamond <=1:
                    change_of_diamond = 0
                else:
                    change_of_diamond = 1
                if picked[4] == last_diamond:
                    exact_diamond = last_diamond
                else:
                    exact_diamond = -1
            mining_time_bak = mining_time
            if mining_time<=400:
                mining_time=-1
            this_lowest = 12345678
            this_highest = -12345678
            this_yield = get_yield_by_picked(picked)
            #
            # change key words
            if j>0:
                #X[change_of_diamond].append(sum_log_yield/(j+5))
                #Y[change_of_diamond].append(this_yield)
                Z[change_of_diamond].loc[len(Z[change_of_diamond])] = [np.log(this_yield+1.0), sum_log_yield/(j+5)]
            #
            last_gold = picked[2]
            last_diamond = picked[4]
            last_lowest = this_lowest
            last_highest = this_highest
            last_yield = this_yield
            sum_log_yield += np.log( this_yield+1.0 )
    
    
    a1 = []
    a2 = []
    b = []
    # change range / num of params
    for j in range(-1,2):
        result = get_median_regression(Z[j], 'y', ['est'])
        a1.append(result.params[1])
        b.append(result.params[0])
    print('a1=',a1)
    print('b=',b)

In [3]:
def DiamondChange(last_total_bid, last_yield, s1, s2, s3, change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond):
    a1= [0.7578083525385929, 1.0421009883606966, 0.7018528271366375]
    b= [0.3337297612939577, -0.17454259332580013, 3.0712839991918104]
    if last_yield == -1:
        s1=5
        s2=5.0*np.log(101.0)
        s3=0
        res = np.log(101.0)
    else:
        s1 = s1 + 1
        now = np.log( last_yield+1 )
        s2 = s2 + now
        x = s2/s1
        # change here
        state = change_of_diamond+1
        res = a1[state] * x + b[state]
    if change_of_diamond == 1:
        res = res + 0.2
    bid = np.exp(res)-1.0
    #bid = int((np.exp(s2/s1)-1.0)/7 + 0.5)
    return (bid,s1,s2,s3)

In [ ]:
def DiamondChange_improve_estimation(last_total_bid, last_yield, s1, s2, s3, change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond):
    if last_yield == -1:
        s1=5
        s2=5.0*np.log(101.0)
    else:
        s1 = s1 + 1
        now = np.log( last_yield+1 )
        s2 = s2 + now
    res = s2/s1
    bid = np.exp(res)-1.0
    #bid = int((np.exp(s2/s1)-1.0)/7 + 0.5)
    return (bid,s1,s2,s3)

In [28]:
player_lst = [NoOccu]*0 + [Fifty] + [GoldChange] + [DiamondChange] + [DiamondExact] + [MiningTime] + [MiningFirst] + [LowHigh]
random.seed(423)
Simulation_I(player_lst, num_mines = 600)
# change_of_gold, mining_time, mining_time_first, change_of_diamond,fifty_yields, lowest, highest, exact_diamond

score [ 9.31358413 12.39088254 16.12063175 12.41840317 11.47416508  9.49593016
 14.14759365]
mae [33.65230794 31.18958095 29.34494286 32.13521587 31.67137778 34.53569524
 31.83364444]
diamond_signal_cc
[[  123  3766  1432]
 [ 2199 28327  3819]
 [ 3038  2185   111]]
diamond_signal_mae_ave
[[  4.67828107  13.2470981  115.91081405]
 [  6.54934061  15.46488207 112.308944  ]
 [ 10.87124988  56.57639752 594.99356499]]
diamond_signal_mae_total
[[   575.42857143  49888.57142857 165984.28571429]
 [ 14402.         438073.71428571 428907.85714286]
 [ 33026.85714286 123619.42857143  66044.28571429]]
score [ 9.88489297 12.77662472 16.42377234 12.90257234 12.06696599  8.4905263
 13.0527263 ]
mae [31.9099619  29.66254444 27.84806667 30.44325397 29.71126984 34.23549683
 31.4961127 ]
diamond_signal_cc
[[  249  7162  2735]
 [ 4306 58299  7183]
 [ 5663  4201   202]]
diamond_signal_mae_ave
[[  4.71543316  13.31108629 113.58218856]
 [  6.4290359   15.61853058 104.88657743]
 [ 11.16667087  58.28003537 492.4

In [27]:
# change_of_gold, mining_time, mining_time_first, change_of_diamond, fifty_yields, exact_diamond, (lowest, highest)
def Simulation_I(player_lst, num_mines = 600):
    n = len(player_lst)
    sv = {}
    for i in range(n):
        for j in range(3):
            sv[(i,j)]=0
    score = np.zeros(n)
    mae = np.zeros(n)
    change_of_diamond_cc=np.zeros((3,3),dtype=int)
    mae_cases = np.zeros((3,3))
    for i in range(num_mines):
        game_value = get_game_value()
        #med = get_median_by_ywlst(game_value)
        last_yield = -1
        last_total_bid = -1
        #
        for j in range(150):
            picked, mining_time_first, mining_time = sample_picked_once_with_time(game_value)
            if j==0:
                change_of_gold = 0
                change_of_diamond = 0
                fifty_yields = []
                for k in range(50):
                    fifty_yields.append(sample_yield_once(game_value))
                last_lowest = 1e9
                last_highest = -1e9
                exact_diamond = -1
                last_change_of_diamond=0
            else:
                fifty_yields = []
                change_of_gold = picked[2] - last_gold
                if change_of_gold<-3:
                    change_of_gold = -2
                elif change_of_gold<=-1:
                    change_of_gold = -1
                elif change_of_gold<=0:
                    change_of_gold = 0
                elif change_of_gold<=3:
                    change_of_gold = 1
                else:
                    change_of_gold = 2
                last_change_of_diamond = change_of_diamond
                change_of_diamond = picked[4] - last_diamond
                if change_of_diamond<-1:
                    change_of_diamond = -1
                elif change_of_diamond <=1:
                    change_of_diamond = 0
                else:
                    change_of_diamond = 1
                if picked[4] == last_diamond:
                    exact_diamond = last_diamond
                else:
                    exact_diamond = -1
            mining_time_bak = mining_time
            if mining_time<=400:
                mining_time=-1
            this_lowest = 12345678
            this_highest = -12345678
            this_yield = get_yield_by_picked(picked)
            bid = np.zeros(n, dtype=int)
            for k in range(n):
                if this_lowest< last_lowest:
                    lowest = this_lowest
                else:
                    lowest = -1
                if this_highest>last_highest:
                    highest = this_highest
                else:
                    highest = -1
                now, sv[(k,0)] , sv[(k,1)], sv[(k,2)] = player_lst[k]( last_total_bid, last_yield, sv[(k,0)] , sv[(k,1)], sv[(k,2)], \
                                                                     change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond)
                
                if now<0:
                    now=0
                bid[k]=now
                this_lowest = min(this_lowest,now)
                this_highest = max(this_highest,now)
            last_gold = picked[2]
            last_diamond = picked[4]
            last_lowest = this_lowest
            last_highest = this_highest
            last_yield = this_yield
            last_total_bid = np.average(bid)
            if last_total_bid<=last_yield:
                score = score+bid
            else:
                score = score+np.average(bid)-bid
            mae = mae + np.abs(this_yield-bid)
            
            change_of_diamond_cc[last_change_of_diamond+1,change_of_diamond+1]+=1
            mae_cases[last_change_of_diamond+1,change_of_diamond+1]+=np.abs(this_yield-bid[2])
        if i%300==299:
            print('score', score/(7.0*150.0*(i+1)))
            print('mae', mae/(7.0*150.0*(i+1)))
            print('diamond_signal_cc')
            print(change_of_diamond_cc)
            print('diamond_signal_mae_ave'  )
            print(np.array(mae_cases)/np.array(change_of_diamond_cc)/7.0)
            print('diamond_signal_mae_total'  )
            print(np.array(mae_cases)/7.0)

In [4]:
# change_of_gold, mining_time, mining_time_first, change_of_diamond, fifty_yields, exact_diamond, (lowest, highest)
def Simulation(player_lst, num_mines = 600):
    n = len(player_lst)
    sv = {}
    for i in range(n):
        for j in range(3):
            sv[(i,j)]=0
    score = np.zeros(n)
    mae = np.zeros(n)
    for i in range(num_mines):
        game_value = get_game_value()
        #med = get_median_by_ywlst(game_value)
        last_yield = -1
        last_total_bid = -1
        #
        for j in range(150):
            picked, mining_time_first, mining_time = sample_picked_once_with_time(game_value)
            if j==0:
                change_of_gold = -1
                change_of_diamond = -1
                fifty_yields = []
                for k in range(50):
                    fifty_yields.append(sample_yield_once(game_value))
                last_lowest = 1e9
                last_highest = -1e9
                exact_diamond = -1
            else:
                fifty_yields = []
                change_of_gold = picked[2] - last_gold
                if change_of_gold<-3:
                    change_of_gold = -2
                elif change_of_gold<=-1:
                    change_of_gold = -1
                elif change_of_gold<=0:
                    change_of_gold = 0
                elif change_of_gold<=3:
                    change_of_gold = 1
                else:
                    change_of_gold = 2
                change_of_diamond = picked[4] - last_diamond
                if change_of_diamond<-1:
                    change_of_diamond = -1
                elif change_of_diamond <=1:
                    change_of_diamond = 0
                else:
                    change_of_diamond = 1
                if picked[4] == last_diamond:
                    exact_diamond = last_diamond
                else:
                    exact_diamond = -1
            mining_time_bak = mining_time
            if mining_time<=400:
                mining_time=-1
            this_lowest = 12345678
            this_highest = -12345678
            this_yield = get_yield_by_picked(picked)
            bid = np.zeros(n, dtype=int)
            for k in range(n):
                if this_lowest< last_lowest:
                    lowest = this_lowest
                else:
                    lowest = -1
                if this_highest>last_highest:
                    highest = this_highest
                else:
                    highest = -1
                now, sv[(k,0)] , sv[(k,1)], sv[(k,2)] = player_lst[k]( last_total_bid, last_yield, sv[(k,0)] , sv[(k,1)], sv[(k,2)], \
                                                                     change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond)
                if k==7:
                    print(picked, mining_time_bak)
                    print(this_yield)
                    print(last_total_bid, last_yield, sv[(k,0)] , sv[(k,1)], sv[(k,2)], \
                                                                     change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond)
                if now<0:
                    now=0
                bid[k]=now
                this_lowest = min(this_lowest,now)
                this_highest = max(this_highest,now)
            last_gold = picked[2]
            last_diamond = picked[4]
            last_lowest = this_lowest
            last_highest = this_highest
            last_yield = this_yield
            last_total_bid = np.average(bid)
            if last_total_bid<=last_yield:
                score = score+bid
            else:
                score = score+np.average(bid)-bid
            mae = mae + np.abs(this_yield-bid)
        if i%300==299:
            print('score', score/(7.0*150.0*(i+1)))
            print('mae', mae/(7.0*150.0*(i+1)))

In [5]:
# change_of_gold, mining_time, mining_time_first, change_of_diamond, fifty_yields, exact_diamond, (lowest, highest)
def Simulation_G(player_lst):
    n = len(player_lst)
    sv = {}
    for i in range(n):
        for j in range(3):
            sv[(i,j)]=0
    score = np.zeros(n)
    mae = np.zeros(n)
    for i in range(1500):
        game_value = get_game_value()
        #med = get_median_by_ywlst(game_value)
        last_yield = -1
        last_total_bid = -1
        #
        for j in range(150):
            picked, mining_time_first, mining_time = sample_picked_once_with_time(game_value)
            if j==0:
                change_of_gold = -1
                change_of_diamond = -1
                fifty_yields = []
                for k in range(50):
                    fifty_yields.append(sample_yield_once(game_value))
                last_lowest = 1e9
                last_highest = -1e9
                exact_diamond = -1
            else:
                fifty_yields = []
                change_of_gold = picked[2] - last_gold
                if change_of_gold<-3:
                    change_of_gold = -2
                elif change_of_gold<=-1:
                    change_of_gold = -1
                elif change_of_gold<=0:
                    change_of_gold = 0
                elif change_of_gold<=3:
                    change_of_gold = 1
                else:
                    change_of_gold = 2
                change_of_diamond = picked[4] - last_diamond
                if change_of_diamond<-1:
                    change_of_diamond = -1
                elif change_of_diamond <=1:
                    change_of_diamond = 0
                else:
                    change_of_diamond = 1
                if picked[4] == last_diamond:
                    exact_diamond = last_diamond
                else:
                    exact_diamond = -1
            mining_time_bak = mining_time
            if mining_time<=400:
                mining_time=-1
            this_lowest = 12345678
            this_highest = -12345678
            this_yield = get_yield_by_picked(picked)
            bid = np.zeros(n, dtype=int)
            for k in range(n):
                if this_lowest< last_lowest:
                    lowest = this_lowest
                else:
                    lowest = -1
                if this_highest>last_highest:
                    highest = this_highest
                else:
                    highest = -1
                now, sv[(k,0)] , sv[(k,1)], sv[(k,2)] = player_lst[k]( last_total_bid, last_yield, sv[(k,0)] , sv[(k,1)], sv[(k,2)], \
                                                                     change_of_gold, mining_time, mining_time_first, change_of_diamond,\
                                                                      fifty_yields, lowest, highest, exact_diamond)
                
                if now<0:
                    now=0
                if k==2:
                    now=this_yield
                bid[k]=now
                this_lowest = min(this_lowest,now)
                this_highest = max(this_highest,now)
            last_gold = picked[2]
            last_diamond = picked[4]
            last_lowest = this_lowest
            last_highest = this_highest
            last_yield = this_yield
            last_total_bid = np.average(bid)
            if last_total_bid<=last_yield:
                score = score+bid
            else:
                score = score+np.average(bid)-bid
            mae = mae + np.abs(this_yield-bid)
        if i%300==299:
            print('score', score/(7.0*150.0*(i+1)))
            print('mae', mae/(7.0*150.0*(i+1)))